# Laboratoire 5 - PolyRISC

Ce *notebook* constitue le **cahier de réponses du Laboratoire 5**. En parallèle de l'énoncé de laboratoire `labo5.pdf`, vous utiliserez ce *notebook* pour décrire votre démarche et pour interagir avec votre design matériel grâce au *framework* PYNQ.

---

# Partie 0 : Préambule technique

***Cette Partie 0 introduit toutes les fonctions utilitaires nécessaires à la communication avec votre implémentation matérielle du PolyRISC. Pour les plus curieux, le fonctionnement du système fait l'objet de descriptions très détaillées. Nul besoin de comprendre tous les détails pour compléter le laboratoire. Cependant, ATTENTION : IL FAUDRA EXÉCUTER LES CELLULES DE CETTE PARTIE À CHAQUE FOIS QUE VOUS VOUDREZ VALIDER UN NOUVEAU DESIGN. Une cellule à la fin de cette partie vous décrit comment effectuer le tout en une seule fois.***

## Survol du design du système

![Schéma du système (à importer dans un dossier `images`)](images/lab5-schema.png)

L'implémentation de ce processeur RISC comprend plusieurs composants, que vous pouvez voir instanciés dans le _top-level_ `top.vhd`. Cette section les présente rapidement ainsi que leur utilité. *Vous n'avez pas besoin de la comprendre dans le détail pour mener à bien le laboratoire, à part pour le PolyRISC*. Les modules sont présentés dans l'ordre d'instanciation dans `top.vhd`.
- `power_on_reset` : Comme vous avez pu le voir dans les laboratoires précédents, un _power-on reset_ est toujours utile pour s'assurer que tous les composants sont réinitialisés lors de la mise sous tension du système. C'est le but de ce composant, qui met son pin de sortie à `'1'` pendant un certain nombre de ticks d'horloge immédiatement après le démarrage.
- `PolyRISC` : Le PolyRISC. C'est une implémentation en FPGA d'un processeur RISC-V basique, tel que vous l'avez vu en cours. Ici, la mémoire des instructions est externe : le PolyRISC a une sortie `o_inst_addr` dirigée par son compteur de programme pour adresser la mémoire des instructions, et une entrée `i_inst` pour lire l'instruction présente dans la mémoire à l'adresse demandée. En outre, il a une entrée et une sortie GPIO, accompagnées de leurs drapeaux de validité, pour pouvoir prendre des entrées et renvoyer des résultats. Ces deux ports sont reliés au GPIO0 du Zynq. Le fichier `PolyRISC_v2.vhd` contient l'implémentation du PolyRISC, et le fichier `PolyRISC_utils_pkg.vhd` contient les déclarations de constantes, types et autres fonctions utilitaires.
- `PolyRISC_ROM` : La mémoire des instructions du PolyRISC. Elle fonctionne comme une ROM, c'est-à-dire que le PolyRISC peut uniquement lire dedans, mais pas y écrire. Les ordinateurs ne fonctionnent plus comme ça de nos jours, mais cette implémentation est plus simple à comprendre et utiliser. Pour pouvoir changer de programme sans re-synthétiser l'intégralité du design, nous allons utiliser PYNQ pour programmer cette ROM avec le contenu d'un buffer alloué en RAM, et rempli par vous depuis ce notebook.
- `AXIS2ROM` : Pour pouvoir programmer la ROM depuis ce notebook Python, nous utilisons le framework PYNQ (et l'API sous-jacente, XRT) qui établissent des connexions avec le Zynq grâce à des bus AXI. Ici, le composant AXI-DMA (Direct Memory Access) instancié dans le _block design_ permet de donner un accès direct à des buffers en mémoire RAM à notre design FPGA, via un bus AXI-Stream. Il faut donc un composant qui puisse recevoir un flux AXI-Stream, et écrire les données reçues dans la ROM. Il s'agit de ce composant. Son implémentation est très basique : dès qu'il reçoit des données valides sur son entrée AXI, il active l'entrée _write enable_ de la ROM et écrit les données reçues l'une après l'autre, en incrémentant l'adresse. Il faut donc le réinitialiser pour réécrire le contenu de la ROM depuis l'adresse `0x0`.
- `zynq_wrapper` : Comme d'habitude, le Zynq est instancié pour permettre la communication avec PYNQ. Ici, il a plusieurs interfaces.
    - GPIO0 (`from_zynq_data` et `to_zynq_data`) est utilisée comme l'interface d'échanges de données entre le PolyRISC et le processeur ARM : l'entrée GPIO du PolyRISC et connectée à la sortie du GPIO1 du Zynq, et inversement. C'est ce qui nous permet d'écrire sur l'entrée GPIO du PolyRISC directement depuis ce notebook Python, encore une fois à travers un bus AXI.
    - GPIO1 (`from_zynq_control`) est utilisée comme l'interface de contrôle du design FPGA. 
        - Le bit 0 (bit de poids le plus faible) est connecté au pin `i_GPIO_valide` du PolyRISC, qui lui indique que la valeur passée sur son GPIO est valide et a pour effet d'incrémenter le compteur de programme.
        - Le bit 1 est utilisé comme reset du PolyRISC, qui a pour effet de remettre son compteur de programme, sa sortie GPIO et ses registres à 0.
        - Le bit 2 est utilisé comme reset du convertisseur AXI-Stream vers ROM, qui remet son adresse à 0 pour permettre de réécrire le contenu de la ROM depuis l'adresse 0 (lorsqu'on veut changer le programme par exemple).

    - `from_zynq` est l'interface AXI-Stream qui permet au composant AXI-DMA de transférer les contenus de la RAM à l'instance `AXIS2ROM` via AXI-Stream

## Imports et détails techniques
On commence par importer `pynq` qui servira à interagir avec le design programmé sur la carte. Nous devons programmer le FPGA à travers le _framework_ PYNQ pour qu'il soit informé de l'architecture de notre design. Cela facilite aussi les interactions avec le PolyRISC.

In [1]:
import pynq
import numpy as np
# Largeur des bus
GPIO_W = 32

`Overlay` est la classe que `pynq` utilise pour représenter un design matériel. On programme le Zynq 7020 en instanciant un `Overlay` à partir d'un _bitstream_ et d'un _hardware handoff_ (`.hwh`). Cela permet :
1. de programmer la logique programmable depuis ce notebook
2. d'informer l'API sous-jacente des composants disponibles et de leur agencement dans notre design, pour qu'elle puisse communiquer avec.

Vous n'avez pas à vous soucier des `Overlay`, des fonctions utilitaires sont implémentées plus bas pour vous permettre d'interagir facilement avec.

In [2]:
# Le hardware handoff top.hwh est chargé implicitement à
# partir du bitstream. pynq.Overlay renvoie une erreur s'il
# ne trouve pas de fichier portant le même nom que son
# argument, avec '.bit' remplacé par '.hwh'
ol = pynq.Overlay("top.bit")

L'overlay contient la description des éléments présents dans le _block design_ que vous avez créé en sourçant le fichier `create_zynq_wrapper.tcl`. Vous pouvez les lister avec la commande :

In [3]:
# On voit que les blocs présents sur le block design sont bien
# représentés dans la structure Overlay. Cette cellule est là
# à titre informatif uniquement.
ol.ip_dict.keys()

dict_keys(['axi_gpio_0', 'axi_gpio_1', 'axi_dma_0', 'processing_system7_0'])

Vous remarquerez qu'on ne voit pas de mention de `top`, ou `PolyRISC_inst`. C'est parce que nos designs ont été synthétisés à partir de fichiers source en VHDL, et non dans le flot de conception de diagramme en blocs de Vivado. Mais cela n'a pas d'importance.

### Instructions prédéfinies
Deux instructions prédéfinies vous sont données : `NOP` et `STOP`.

`NOP` (No OPeration) consiste en un branchement toujours faux. Cette instruction ne fait rien : le compteur de programme est incrémenté et c'est la seule chose qui est réalisée sur le tick d'horloge courant.

`STOP` consiste en un branchement toujours vrai avec un incrément de 0. Comme son nom l'indique, `STOP` stoppe le processeur : le compteur de programme n'incrémente pas et le processeur reste bloqué sur cette instruction jusqu'au prochain reset.

Reportez-vous au matériel de cours du chapitre 9 pour bien comprendre le lien entre branchement toujours / jamais et `STOP` / `NOP`.

In [4]:
# Instructions prédéfinies
# NOP = branchement_jamais_0_0_0
NOP  = 0b10_0111_00000_00000_0000000000000000
# STOP = branchement_toujours_0_0_0
STOP = 0b10_0110_00000_00000_0000000000000000

### Communication avec le design
Pour donner un programme à exécuter au PolyRISC, on utilise les fonctionnalités AXI-DMA de `pynq`. `pynq.allocate` permet d'allouer un buffer qui est mis à disposition du FPGA, via un bus AXI. Le module `AXIS2ROM` du projet sert cette fonction : il reçoit des données depuis le bus AXI-Stream et remplit la ROM simulée en FPGA (module `PolyRISC_ROM`), qui est connectée à l'entrée d'instructions et au compteur de programme du PolyRISC.

In [5]:
# Buffer utilisé pour transférer la mémoire des
# instructions vers le module ROM
in_buf = pynq.allocate(shape=(32,), dtype=np.uint32)
# Le buffer obtenu est exposé à l'utilisateur sous la forme
# d'un array numpy, avec toutes les fonctionnalités qui
# en découlent

# On initialise le buffer avec des NOP, pour éviter
# de donner de mauvaises instructions au processeur
# par erreur
for i in range(len(in_buf)):
    in_buf[i] = NOP

Comme le but de ce laboratoire n'est **pas** de vous former à l'utilisation des `Overlay` sous PYNQ, les cellules ci-dessous implémentent un certain nombre de fonctions pour vous permettre d'interagir facilement avec le design. *Vous n'avez pas besoin de les comprendre pour mener à bien ce laboratoire*.

### Utilitaires
Les cellules suivantes définissent toutes les fonctions dont vous aurez besoin pour tester votre design implémenté sur la carte.

In [6]:
# Masque pour l'écriture des GPIO du Zynq
# On écrit tous les bits
gpio_mask = 0xffffffff

#### Reset AXIS2ROM
Le bit 2 (troisième à partir du bit de poids le plus faible) du GPIO1 du processeur ARM est connecté au pin `reset` du module `AXIS2ROM` (voir l'instance `S2R_inst` dans le `top.vhd`). Pour réinitialiser le convertisseur AXI-Stream vers ROM, ce qui a pour effet de remettre sa sortie d'adresse à 0, on doit donc écrire 4 (`0b100`) sur le GPIO1. On le remet ensuite à 0 pour permettre au module de fonctionner (sinon, il serait réinitialisé en permanence).

In [7]:
def reset_AXIS2ROM(overlay):
    # Le bit 1 du GPIO1 est connecté au reset des
    # composants dans le toplevel
    overlay.axi_gpio_1.channel1.write(0b100, gpio_mask)
    overlay.axi_gpio_1.channel1.write(0b0, gpio_mask)

#### Reset PolyRISC
Le bit 1 du GPIO1 du processeur ARM est relié au pin `reset` du PolyRISC. On procède donc de même que pour `AXIS2ROM` mais en écrivant 2 (`0b10`) sur le GPIO1.

In [8]:
def reset_PolyRISC(overlay):
    # Le bit 1 du GPIO1 est connecté au reset des
    # composants dans le toplevel
    overlay.axi_gpio_1.channel1.write(0b10, gpio_mask)
    overlay.axi_gpio_1.channel1.write(0b0, gpio_mask)

#### Écriture de la ROM via le buffer PYNQ
Les fonctions suivantes initialisent le buffer alloué par PYNQ et écrivent vos instructions dedans. Vous devez passer les instructions sous forme d'une liste de nombres sur maximum 32 bits. En pratique, vous utiliserez
```python
flash(mes_instructions, ol, in_buf)
```
pour écrire votre liste d'instructions dans la ROM d'instructions du PolyRISC.

Astuces : 
- vous pouvez écrire des littéraux en binaire en Python avec le préfixe `0b`
- les _underscore_ (`_`) dans les littéraux binaires sont ignorés

Reportez-vous au programme de calcul des nombres de Fibonacci plus bas pour avoir un exemple d'utilisation de ces fonctions.

In [9]:
def init_instructions(liste_instructions, buffer):
    # Initialiser tout le programme à NOP, par sécurité
    for i in range(len(buffer)):
        buffer[i] = NOP
    # Remplir le buffer passé 2e en argument avec les 
    # valeurs contenues dans le 1er argument
    for i, inst in enumerate(liste_instructions):
        buffer[i] = inst

def flash_ROM(overlay, buffer):
    # Reset le convertisseur AXIS-ROM pour remettre son
    # pointeur d'adresse à 0
    reset_AXIS2ROM(overlay)
    # Reset le PolyRISC pour être sûr que son compteur
    # de programme est bien à 0
    reset_PolyRISC(overlay)
    # Transférer le buffer d'instructions dans la ROM
    # en AXI-Stream
    overlay.axi_dma_0.sendchannel.transfer(buffer)

def flash(liste_instructions, overlay, buffer):
    # Raccourci pour appeler init_instructions et flash_ROM
    # en une seule commande
    init_instructions(liste_instructions, buffer)
    flash_ROM(overlay, buffer)

#### Lancement du programme
Enfin, vous pouvez lancer le programme écrit dans la ROM sur l'entrée voulue en appelant la fonction `test_PolyRISC` ci-dessous. Cette fonction a pour effet de réinitialiser le PolyRISC, d'écrire `entree` sur son GPIO, puis appelle `go_PolyRISC` qui allume puis éteint le bit 0 du GPIO1 du Zynq (qui est connecté à l'entrée `i_GPIO_valide` du PolyRISC), ce qui a pour effet de permettre au PolyRISC de lire le GPIO et de poursuivre l'exécution du programme. `test_PolyRISC` retourne la valeur lue sur la sortie GPIO du PolyRISC à l'issue de l'exécution du programme.

In [10]:
# Complément à 2 à la main pour traduire le fait que le
# GPIO est signé
def cplt2(num, w=32):
    out = format(num, f"0{w}b")
    if out[0] == "1":
        binout = format(num - 1, f"0{w}b")
        flip2sc = "".join([str(1 - int(i)) for i in binout])
        return - int(flip2sc, 2)
    else:
        return int(out, 2)

def go_PolyRISC(overlay):
    # Le bit 0 du GPIO1 est connecté au pin 
    # i_GPIO_valide du PolyRISC
    overlay.axi_gpio_1.channel1.write(0b01, gpio_mask)
    overlay.axi_gpio_1.channel1.write(0b00, gpio_mask)

def test_PolyRISC(entree, overlay):
    # Reset le PolyRISC
    reset_PolyRISC(overlay)
    # Envoyer l'entree au GPIO du PolyRISC
    # Le GPIO0 du zynq est connecté au GPIO du PolyRISC,
    # et on écrit dedans depuis ce notebook via la communication
    # AXI offerte par XRT, l'infrastructure de PYNQ
    # Le channel 1 du GPIO0 du zynq est l'entrée du PolyRISC,
    # on écrit dedans pour écrire sur l'entrée du PolyRISC
    overlay.axi_gpio_0.channel1.write(entree, gpio_mask)
    # Lancer le calcul sur PolyRISC
    go_PolyRISC(overlay)
    # Le channel 2 du GPIO0 est connecté à la sortie du PolyRISC
    # Techniquement, on devrait attendre que PolyRISC indique
    # que sa sortie est valide avant de lire le GPIO0, mais dans
    # le cas présent les instructions en Python sont suffisamment
    # lentes pour laisser le temps au PolyRISC de calculer
    out = overlay.axi_gpio_0.channel2.read()
    return cplt2(out, w=GPIO_W)

### Voilà pour la Partie 0. **Avant de valider un design matériel avec un nouveau `top.bit` et `top.hwh`** : 
## -> CLIQUEZ ICI et faites `Cell > Run All Above`

# Partie 1 : Découverte du PolyRISC

### Question 1 : Insérez ci-dessous une capture d'écran du chronogramme de simulation pour la suite de Fibonacci.
![CaptureChronogramme](images/lab5Q1.png)

### Question 2 : Décrivez en quelques mots comment vous avez fait en sorte que le banc d'essai écrive les résultats dans un fichier texte.

Nous avons d’abord ouvert le fichier de sortie en mode écriture :

file_open(fichier_sorties, "C:/Users/noeju/OneDrive/Desktop/INF3500/lab/lab5/labo5-2429561-2433385/src/fibonacci_res.txt", write_mode);

Ensuite, une fois la sortie reliée à l’entrée courante, le résultat est stocké dans une variable integer nommée data. Après, une ligne contenant data est écrite à l’aide de la fonction write(), puis cette ligne a été enregistrée dans le fichier grâce à writeline(). Voici le code :

data := to_integer(GPIO_out);
write(line_out, data);
writeline(fichier_sorties, line_out);

### Question 3 : Après avoir téléversé vos `top.bit` et `top.hwh`, exécutez la cellule Python ci-dessous pour vérifier matériellement que le PolyRISC calcule bien le nombre de Fibonacci.

La sortie de la cellule Python correspond bien aux résultats attendus, soit:
0
1
2
5
21
55
144
6765
196418
24157817
433494437
1134903170

In [11]:
# 1. CODE MACHINE
# Calcule le nombre de Fibonacci du rang passé en entrée sur le GPIO
instructions_fibonacci = [
    0b11_0010_00000_00000_0000000000000000, # 0: R0 := i_GPIO
    0b01_0001_00001_00000_0000000000000000, # 1: R1 := $0
    0b01_0001_00011_00000_0000000000000000, # 2: R3 := $0
    0b01_0001_00100_00000_0000000000000001, # 3: R4 := $1
    0b10_0000_00001_00000_0000000000000110, # 4: si R1 = R0 goto CP + 6
    0b00_0010_00101_00011_0000000000000100, # 5: R5 := R3 + R4
    0b00_0000_00011_00100_0000000000000000, # 6: R3 := R4
    0b00_0000_00100_00101_0000000000000000, # 7: R4 := R5
    0b01_0010_00001_00001_0000000000000001, # 8: R1 := R1 + 1
    0b10_0110_00000_00000_1111111111111011, # 9: toujours goto CP + -5
    0b11_0011_00011_00000_0000000000000000, # A: o_GPIO := R3
    STOP,
]

# 2. ECRITURE DANS LA MEMOIRE D'INSTRUCTIONS
# On écrit le programme dans le buffer PYNQ alloué précédemment, et
# on transfère ce buffer dans la ROM du PolyRISC (`flash` réalise 
# ces deux actions)
flash(instructions_fibonacci, ol, in_buf)

# 3. TEST DU POLYRISC
# Les résultats devraient être les mêmes que ceux du test-bench
vecteur_test = [0, 1, 3, 5, 8, 10, 12, 20, 27, 37, 43, 45]
for t in vecteur_test:
    print(test_PolyRISC(t, ol))

0
1
2
5
21
55
144
6765
196418
24157817
433494437
1134903170


# Partie 2 : Extension du PolyRISC

### Question 4 : Décrivez en quelques mots comment vous avez ajouté les nouvelles opérations demandées au PolyRISC.

Premièrement, dans le fichier PolyRISC_utils_pkg.vhd, nous avons défini les constantes des opérations, de la manière suivante:

CONSTANT AmulB : NATURAL := 11;
CONSTANT Adiv2 : NATURAL := 12;

Deuxièmement, nous avons ajouté les opérations à effectuer dans l’UAL en fonction de ces nouvelles constantes. Nous avons fait cela dans le fichier PolyRISC_v2.vhd, de la manière suivante:

A(POLYRISC_GPIO_W/2 - 1 downto 0) * B(POLYRISC_GPIO_W/2 -1 downto 0) when AmulB,
A / 2    when Adiv2,

Lors de la multiplication de A et B, comme mentionné dans les instructions du laboratoire, nous prenons les POLYRISC_GPIO_W / 2 bits de poids faible de A et B afin d’éviter un débordement de la taille POLYRISC_GPIO_W.

### Question 5 : Après avoir téléversé vos `top.bit` et `top.hwh`, exécutez la cellule Python ci-dessous pour vérifier matériellement que le PolyRISC étendu fonctionne comme attendu.

La sortie de la cellule Python correspond bien aux résultats attendus, soit:

Vecteur de test : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

RC := RA * RB (avec RB = 4)
Résultat : [4, 8, 12, 16, 20, 24, 28, 32, 36, 40]

RC := RA / 2
Résultat : [0, 1, 1, 2, 2, 3, 3, 4, 4, 5]

In [12]:
# 0. VECTEUR DE TEST
vec = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(f"Vecteur de test : {vec}")

###

# 1. TEST DE LA MULTIPLICATION RC := RA * RB 
# Test en multipliant l'entrée par 4
test_mul_reg = [
    0b11_0010_00000_00000_0000000000000000, # R0 := i_GPIO
    0b01_0001_00001_00000_0000000000000100, # R1 := $4
    0b00_1011_00010_00000_0000000000000001, # R2 := R0 * R1
    0b11_0011_00010_00000_0000000000000000  # o_GPIO := R2
]
# Écriture de la ROM
flash(test_mul_reg, ol, in_buf)
# Test
print("\nRC := RA * RB (avec RB = 4)")
out = []
for t in vec:
    out.append(test_PolyRISC(t, ol))
print(f"Résultat : {out}")

###

# 2. TEST DE LA DIVISION RC := RA / 2
test_div = [
    0b11_0010_00000_00000_0000000000000000, # R0 := i_GPIO
    0b01_1100_00000_00000_0000000000000000, # R0 := R0 / 2
    0b11_0011_00000_00000_0000000000000000  # o_GPIO := R0
]
# Écriture de la ROM
flash(test_div, ol, in_buf)
# Test
print("\nRC := RA / 2")
out = []
for t in vec:
    out.append(test_PolyRISC(t, ol))
print(f"Résultat : {out}")

Vecteur de test : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

RC := RA * RB (avec RB = 4)
Résultat : [4, 8, 12, 16, 20, 24, 28, 32, 36, 40]

RC := RA / 2
Résultat : [0, 1, 1, 2, 2, 3, 3, 4, 4, 5]


# Partie 3 : Simulation de la recherche dichotomique

### Question 6 : Écrivez ci-dessous votre code pseudo-assembleur correspondant à l'algorithme de recherche dichotomique présenté dans l'énoncé. Remplissez ensuite le tableau décrivant l'utilisation que vous faites des registres.


```
R0 := GPIO_in
R1 := $0x7FFF
R2 := $0
R5 := $16
R6 := $0

si R5 = R6 aller à CP + 10

R3 := R1 + R2
R3 := R3 / 2
R4 := R3 * R3

si R4 <= R0 aller à CP + 3
R1 := R3
aller à CP + 2
R2 := R3

R5 := R5 - $1
aller à CP - 9

GPIO_out := R3
STOP
```


|Registre |Contenu |
|---------|--------|
|`fichierRegistres[0]` |entrée nombre, lue depuis GPIO_in |
|`fichierRegistres[1]` |haut, initialisé à 0x7FFF |
|`fichierRegistres[2]` |bas, initialisé à 0|
|`fichierRegistres[3]` |pivot = (haut + bas) / 2 |
|`fichierRegistres[4]` |carre = pivot * pivot |
|`fichierRegistres[5]` |compteur initialisé à 16 |
|`fichierRegistres[6]` |constante 0  pour le test de fin de boucle |



### Question 7 : Insérez ci-dessous une capture d'écran du chronogramme de simulation pour la recherche dichotomique.

![Chronogramme dichotomie](images/chrono_dichotomie.png)

---

# Partie 4 : Implémentation de la recherche dichotomique

### Question 8 : Complétez la cellule Python ci-dessous en écrivant dans `notre_programme` le code machine correspondant au pseudo-assembleur que vous avez écrit à la **Question 6**.

In [13]:
# À COMPLÉTER AVEC VOTRE CODE MACHINE
notre_programme = [
    0b11_0010_00000_00000_0000000000000000,  # R0 := i_GPIO
    0b01_0001_00001_00000_0111111111111111,  # R1 := 0x7FFF
    0b01_0001_00010_00000_0000000000000000,  # R2 := 0
    0b01_0001_00101_00000_0000000000010000,  # R5 := 16
    0b01_0001_00110_00000_0000000000000000,  # R6 := 0
    0b10_0000_00110_00101_0000000000001010,  # si R5 = R6 aller à FIN
    0b00_0010_00011_00001_0000000000000010,  # R3 := R1 + R2
    0b00_1100_00011_00011_0000000000000000,  # R3 := R3 / 2
    0b00_1011_00100_00011_0000000000000011,  # R4 := R3 * R3
    0b10_0100_00000_00100_0000000000000011,  # si R4 <= R0 aller à BAS
    0b00_0000_00001_00011_0000000000000000,  # R1 := R3
    0b10_0110_00000_00000_0000000000000010,  # aller à SUITE
    0b00_0000_00010_00011_0000000000000000,  # R2 := R3
    0b01_0011_00101_00101_0000000000000001,  # R5 := R5 - 1
    0b10_0110_00000_00000_1111111111110111,  # aller à BOUCLE
    0b11_0011_00011_00000_0000000000000000,  # o_GPIO := R3
]

### Question 9 : Complétez la cellule Python ci-dessous en écrivant dans `nos_valeurs` vos vecteurs de test utilisés dans votre banc d'essai plus tôt, afin de tester votre racine carrée.

In [14]:
# À COMPLÉTER AVEC VOS VECTEURS DE TEST
nos_valeurs = [0, 1, 2, 4, 9, 16, 25, 49, 100, 144, 256, 400, 625, 1000, 1024, 2500, 4096,
5000, 8000, 10000, 16000, 20000, 25000, 30000, 32767]

### Question 10 : Avec vos `top.bit` et `top.hwh` déjà téléversés, exécutez la cellule Python ci-dessous pour vérifier matériellement que le PolyRISC calcule bien la racine carrée.

***NB : Assurez-vous de bien avoir exécuté toutes les cellules de la Partie 0 au préalable si ça n'a pas déjà été fait***, ainsi que les cellules des Questions 8 et 9.

In [15]:
# ECRITURE DANS LA MEMOIRE D'INSTRUCTIONS
flash(notre_programme, ol, in_buf)

# TEST DU POLYRISC
out = []
for t in nos_valeurs:
    out.append(test_PolyRISC(t, ol))
print(out)

[0, 1, 1, 2, 3, 4, 5, 7, 10, 12, 16, 20, 25, 31, 32, 50, 64, 70, 89, 100, 126, 141, 158, 173, 181]
